# 🎓 University Admissions Advisor (Local Windows Version)

**Previously on Google Colab → Now adapted for Windows local machine**

### Changes made:
- Removed all Linux-specific commands (`sudo apt-get`, `curl | sh`, `nohup`)
- Fixed PDF path from hardcoded Colab path to local Windows path
- Fixed broken `langchain.chains` imports → `langchain_classic.chains`
- Changed model from `qwen2.5:3b` to the already-installed `qwen2.5:7b`
- Ollama server management now assumes Windows service (runs in system tray)

In [1]:
# ==========================================
# CELL 0: VERIFY OLLAMA IS RUNNING ON WINDOWS
# ==========================================
import requests
import sys

print("--> Checking if Ollama server is running on Windows...")
try:
    resp = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
    if resp.status_code == 200:
        models = [m["name"] for m in resp.json().get("models", [])]
        print(f"   Ollama server is active!")
        print(f"   Available models: {', '.join(models)}")
        
        # Verify required models exist
        has_qwen = any("qwen2.5:7b" in m for m in models)
        has_nomic = any("nomic-embed-text" in m for m in models)
        
        if not has_qwen:
            print("   WARNING: qwen2.5:7b model not found. Run: ollama pull qwen2.5:7b")
        if not has_nomic:
            print("   WARNING: nomic-embed-text model not found. Run: ollama pull nomic-embed-text")
            
        if not has_qwen or not has_nomic:
            print("\n--> Pulling missing models...")
            import subprocess
            if not has_qwen:
                subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=True)
            if not has_nomic:
                subprocess.run(["ollama", "pull", "nomic-embed-text"], check=True)
    else:
        print(f"   ERROR: Unexpected status code {resp.status_code}")
        sys.exit(1)
except requests.ConnectionError:
    print("   ERROR: Cannot reach Ollama at http://127.0.0.1:11434")
    print("   Make sure the Ollama app is running (check system tray / taskbar).")
    print("   If not installed, download from: https://ollama.com/download/windows")
    sys.exit(1)

print("\n[SUCCESS] Ollama server is ready!")

--> Checking if Ollama server is running on Windows...
   Ollama server is active!
   Available models: qwen2.5:7b, phi3:mini, nomic-embed-text:latest, qwen2.5:7b-instruct

[SUCCESS] Ollama server is ready!


In [2]:
# ==========================================
# CELL 1: INSTALL PYTHON DEPENDENCIES
# ==========================================
print("--> Installing required Python packages...")
!pip install -q langchain langchain-ollama chromadb pypdf langchain-community langchain-classic

print("\n[SUCCESS] Dependencies installed!")

--> Installing required Python packages...

[SUCCESS] Dependencies installed!



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ==========================================
# CELL 2: LOAD AND CHUNK PDF
# ==========================================
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Build the PDF path relative to the current working directory
# Launch Jupyter from D:\university_project_demo for this to work
pdf_path = os.path.join(os.getcwd(), "content", "sample_data", "UMD_and_FDU_University_Profile_Report.pdf")

print(f"--> Current working directory: {os.getcwd()}")
print(f"--> Looking for PDF at: {pdf_path}")
print("-" * 50)

if not os.path.exists(pdf_path):
    # Try alternate paths
    alternates = [
        "content/sample_data/UMD_and_FDU_University_Profile_Report.pdf",
        "content\\sample_data\\UMD_and_FDU_University_Profile_Report.pdf",
    ]
    found = False
    for alt in alternates:
        if os.path.exists(alt):
            pdf_path = alt
            found = True
            break
    
    if not found:
        print("\n--> Files in current directory:")
        for item in os.listdir(os.getcwd()):
            print(f"    {item}")
        raise FileNotFoundError(
            f"Could not find the PDF file.\n"
            f"Expected at: {pdf_path}\n"
            f"Make sure you launched Jupyter from D:\\university_project_demo"
        )

print(f"--> Found the file. Ingesting pages...")
loader = PyPDFLoader(pdf_path)
raw_documents = loader.load()
print(f"[INFO] Successfully loaded {len(raw_documents)} pages from the PDF.")

# Split text strategically to protect the tables
print("--> Splitting document pages into structured text chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len
)
processed_chunks = text_splitter.split_documents(raw_documents)

print(f"\n[SUCCESS] Step 2 completed! Generated {len(processed_chunks)} dynamic context chunks.")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5468\3863134467.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


--> Current working directory: d:\university_project_demo
--> Looking for PDF at: d:\university_project_demo\content\sample_data\UMD_and_FDU_University_Profile_Report.pdf
--------------------------------------------------
--> Found the file. Ingesting pages...
[INFO] Successfully loaded 15 pages from the PDF.
--> Splitting document pages into structured text chunks...

[SUCCESS] Step 2 completed! Generated 50 dynamic context chunks.


In [4]:
# ==========================================
# CELL 3: VECTOR DATABASE INITIALIZATION
# ==========================================
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
import warnings

# Suppress minor warnings for a cleaner output
warnings.filterwarnings("ignore")

print("--> Initializing the local Ollama Embedding engine (nomic-embed-text)...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print(f"--> Generating vector embeddings for {len(processed_chunks)} chunks...")
print("--> Storing them in ChromaDB (This might take 10-20 seconds)...")

# Create the vector database from our document chunks
vector_store = Chroma.from_documents(
    documents=processed_chunks,
    embedding=embeddings,
    persist_directory="./chroma_local_db"
)

print("--> Configuring the retrieval engine...")
# Set up the retriever to fetch the top 3 most relevant chunks based on the user's question
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("\n[SUCCESS] Step 3 completed! ChromaDB is loaded and the Retriever is ready.")

--> Initializing the local Ollama Embedding engine (nomic-embed-text)...
--> Generating vector embeddings for 50 chunks...
--> Storing them in ChromaDB (This might take 10-20 seconds)...
--> Configuring the retrieval engine...

[SUCCESS] Step 3 completed! ChromaDB is loaded and the Retriever is ready.


In [5]:
# Verify the installed LangChain package versions
print('--> Verifying LangChain package versions:')
!pip show langchain langchain-core langchain-community langchain-classic

--> Verifying LangChain package versions:
Name: langchain
Version: 1.3.14
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: C:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.4.9
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: C:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-ollama, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk
---
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
Lice

In [ ]:
# ==========================================
# CELL 4: ADVISOR PERSONA & CHAT LOOP (FIXED FOR LANGCHAIN >=1.0)
# ==========================================
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
import warnings

warnings.filterwarnings("ignore")

print("--> Booting the local Qwen 2.5 7B Chat Engine (Temperature = 0.0)...")
# Using the existing qwen2.5:7b model on this machine
llm = ChatOllama(model="qwen2.5:7b", temperature=0.0)

print("--> Injecting the Strategic Admissions System Prompt...")
system_prompt = (
    "You are a warm, helpful, and highly precise University Admissions Advisor. "
    "Your goal is to guide students through their inquiries using ONLY the provided university profile context.\n\n"
    "CONVERSATIONAL TONE RULES:\n"
    "1. Be encouraging, professional, and approachable. Treat the student like a welcome addition to our community.\n"
    "2. Avoid purely mechanical robotic language. Use natural transitions (e.g., \"I'd be happy to break down those costs for you...\").\n"
    "3. Keep your overall responses concise and easy to read for a student scrolling on a screen.\n\n"
    "STRICT FACTUAL CONSTRAINTS:\n"
    "1. Rely EXCLUSIVELY on the provided context. If a fee, requirement, or course is not explicitly stated in the text, "
    "politely respond: \"I want to give you the most accurate information, but that specific detail isn't in my current records.\"\n"
    "2. Never merge or confuse details between the University of Maryland (UMD) and Fairleigh Dickinson University (FDU).\n"
    "3. Present all structural information (tuition lists, deadlines, course names) in clear, well-spaced Markdown formatting.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Assemble the RAG chain components together
print("--> Building the LangChain Retrieval-Augmented Generation infrastructure...")
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("\n[SUCCESS] All systems online! You can now start testing the bot.")
print("=" * 60)

# Interactive loop runner
def test_bot():
    print("\nAsk a question below. Type 'exit' to stop the test.\n")
    while True:
        user_input = input("Student: ")
        if user_input.lower().strip() in ['exit', 'quit']:
            print("Admissions Engine stopped. Testing finished!")
            break
        if not user_input.strip():
            continue

        print("\n--> Fetching relevant document chunks and processing response...")
        try:
            response = rag_chain.invoke({"input": user_input})
            print(f"\nAdvisor Bot:\n{response['answer']}")
            print("-" * 60)
        except Exception as e:
            print(f"\n[ERROR] Could not connect to the model engine: {str(e)}")
            print("Please make sure the Ollama app is running in your system tray.")
            break

# Run the test interactive mode
test_bot()

--> Booting the local Qwen 2.5 7B Chat Engine (Temperature = 0.0)...
--> Injecting the Strategic Admissions System Prompt...
--> Building the LangChain Retrieval-Augmented Generation infrastructure...

[SUCCESS] All systems online! You can now start testing the bot.

Ask a question below. Type 'exit' to stop the test.


--> Fetching relevant document chunks and processing response...

Advisor Bot:
The university we're discussing is the University of Maryland, College Park (UMD). If you have any specific questions about UMD, feel free to ask!
------------------------------------------------------------


In [ ]:
# ==========================================
# CELL 5: QUICK RESTART (WINDOWS)
# ==========================================
# On Windows, Ollama runs as a background service (check system tray).
# No need to manually start the server — just re-run the chat loop.

print("--> Restarting the chat interface...\n")
print("=" * 60)

# Re-run the chat loop we defined in Cell 4
test_bot()